# 01 - Create Scatterer Databases

This notebook generates the source scatterer distributions used throughout the simulation study.

Two databases are constructed:

1. Discrete point-scatterer distributions, consisting of either two or three weighted point scatterers.
2. Matched Gaussian-mixture distributions, constructed by fitting a two-component equal-variance Gaussian mixture model (GMM) to the low-order statistical moments of each discrete distribution.

For each distribution, the notebook records its mean, variance, standard deviation, skewness, kurtosis, and higher standardized moments.

## Setup

This notebook should be immediately runnable from the repository root. The reusable analysis modules are loaded from `src/`.

## Outputs

Running the full notebook creates:

- `delta_database.csv`
- `gmm_database.csv`

in the repository root. These files are used as inputs by
`02_generate_measurements.ipynb`.

> Runtime: The full database contains approximately 688,000 candidate scatterer configurations, and fitting a GMM to every configuration is computationally expensive.

In [ ]:
import sys
import time
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
ROOT    = Path.cwd()
SRC_DIR = ROOT / "src"

if not SRC_DIR.is_dir():
    raise FileNotFoundError("Project root not found. Run this notebook with the repository "
                            "root as the current working directory.")

In [ ]:
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import gmm
import moments as mfl
import scatterers as sfl

# Database Generation

The discrete database samples two- and three-point scatterer configurations over a range of center locations, scatterer separations, and normalized reflectivity weights.

All point-scatterer locations are chosen on the sampled spatial/delay grid so that the discrete distributions can be represented exactly.

In [ ]:
moment_names    = ['mean', 'variance', 'std', 'skew', 'kurtosis', 'inv_kurt', 'hyperskewness', 'hyperkurtosis']
delta_cols      = ['loc_1', 'loc_2', 'loc_3', 'amp_1', 'amp_2', 'amp_3'] + moment_names
gmm_cols        = ['loc_1', 'loc_2', 'loc_3', 'amp_1', 'amp_2', 'amp_3', 'mu1', 'mu2', 'sigma', 'w1'] + moment_names

mid_low         = -0.1
mid_high        = 0.1
mid_step        = 0.02

extent_low      = 0.02
extent_high     = 0.401
extent_step     = 0.02

sample_rate     = 100
scatter_len     = 4
x               = np.round(np.linspace(-scatter_len/2, scatter_len/2, scatter_len*sample_rate, endpoint=False), 3)

DELTA_DB        = ROOT / "delta_database.csv"
GMM_DB          = ROOT / "gmm_database.csv"

### Preallocate storage

The number of configurations is counted before generation so that the complete NumPy arrays can be allocated once rather than repeatedly expanded during the loop.

In [ ]:
num_records     = 0

for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        for amp_1 in np.round(np.arange(.05, .96, .05), 2):
            num_records += 1
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_3 = np.round(mid + extent, 2)
        for spot in np.round(np.arange(.02, extent*2-.01, .02), 2):
            for amp_1 in np.round(np.arange(.05, .91, .05), 2):
                for amp_2 in np.round(np.arange(.05, .96-amp_1, .05), 2):
                    num_records += 1

print('Number of records: ', num_records)

### Generate discrete scatterers and matched GMMs

For each discrete distribution:

1. construct the normalized point-scatterer distribution
2. calculate its statistical moments
3. fit a two-component equal-variance GMM to the target low-order moment statistics
4. evaluate the fitted GMM on the same sampled coordinate grid
5. store both the discrete and GMM distribution parameters

In [ ]:
delta_array = np.zeros((num_records, len(delta_cols)))
gmm_array   = np.zeros((num_records, len(gmm_cols)))
start       = time.time()
idx         = 0

# two deltas loop
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_2 = np.round(mid + extent, 2)
        for amp_1 in np.round(np.arange(.05, .96, .05), 2):
            amp_2               = np.round(1.0 - amp_1, 2)
            delta               = sfl.Delta(x, loc1=loc_1, loc2=loc_2, amp1=amp_1, amp2=amp_2)
            ems                 = mfl.Moments(x, delta.scatter)
            vals                = [loc_1, loc_2, .49, amp_1, amp_2, 0.]
            delta_array[idx]    = np.concatenate((vals, ems.moments))
            target_moments      = [ems.mean, ems.variance, ems.std, ems.skew, ems.kurtosis]
            results, _, _       = gmm.match_moments(target_moments)
            gmm_scatter         = gmm.gmm_pdf(x, results)
            gmm_ems             = mfl.Moments(x, gmm_scatter)
            gmm_vals            = np.concatenate((vals, results))
            gmm_array[idx]      = np.concatenate((gmm_vals, gmm_ems.moments))
            idx += 1
            if idx % 1000 == 0:
                print(idx)

# three deltas loop
for mid in np.round(np.arange(mid_low, mid_high, mid_step), 2):
    for extent in np.round(np.arange(extent_low, extent_high, extent_step), 2):
        loc_1 = np.round(mid - extent, 2)
        loc_3 = np.round(mid + extent, 2)
        for spot in np.round(np.arange(.02, extent*2-.01, .02), 2):
            loc_2 = np.round(spot + loc_1, 2)
            for amp_1 in np.round(np.arange(.05, .91, .05), 2):
                for amp_2 in np.round(np.arange(.05, .96-amp_1, .05), 2):
                    amp_3               = np.round(1.0 - amp_1 - amp_2, 2)
                    delta               = sfl.Delta(x, loc1=loc_1, loc2=loc_2, loc3=loc_3,
                                                    amp1=amp_1, amp2=amp_2, amp3=amp_3)
                    ems                 = mfl.Moments(x, delta.scatter)
                    vals                = [loc_1, loc_2, loc_3, amp_1, amp_2, amp_3]
                    delta_array[idx]    = np.concatenate((vals, ems.moments))
                    target_moments      = [ems.mean, ems.variance, ems.std, ems.skew, ems.kurtosis]
                    results, _, _       = gmm.match_moments(target_moments)
                    gmm_scatter         = gmm.gmm_pdf(x, results)
                    gmm_ems             = mfl.Moments(x, gmm_scatter)
                    gmm_vals            = np.concatenate((vals, results))
                    gmm_array[idx]      = np.concatenate((gmm_vals, gmm_ems.moments))
                    idx += 1
                    if idx % 1000 == 0:
                        print(idx)

delta_db    = pd.DataFrame(delta_array, columns=delta_cols)
gmm_db      = pd.DataFrame(gmm_array, columns=gmm_cols)
end         = time.time()
print("Time taken: ", (end - start)/60)

delta_db.to_csv(DELTA_DB, index=False)
gmm_db.to_csv(GMM_DB, index=False)

# Clean Up GMM Database

GMM parameters are obtained using numerical minimization, which does not always converge to a sufficiently accurate match. Candidate GMMs whose mean, standard deviation, skewness, or kurtosis differs from the target by more than 0.01 are removed from the final database.

In [ ]:
delta_db    = pd.read_csv(DELTA_DB)
gmm_db      = pd.read_csv(GMM_DB)

In [ ]:
gmm_db['good']  = True
tol             = .01
check_cols      = ['mean', 'std', 'skew', 'kurtosis']

for idx, row in gmm_db.iterrows():
    if idx % 1000 == 0:
        print(idx)
    gmm_scatter = sfl.GMM(x, row['mu1'], row['mu2'], row['sigma'], row['w1'])
    moments     = mfl.Moments(gmm_scatter.x, gmm_scatter.scatter)
    stored      = row[check_cols].to_numpy(dtype=float)
    target      = delta_db.loc[idx, check_cols].to_numpy(dtype=float)
    moms_core   = np.array([moments.mean, moments.std, moments.skew, moments.kurtosis])

    if (not np.allclose(moms_core, stored, atol=tol, rtol=0) or not np.allclose(stored, target, atol=tol, rtol=0)):
        gmm_db.loc[idx, 'good'] = False

gmm_db          = gmm_db[gmm_db['good'] == True].drop(columns='good')

print(delta_db.shape)
print(gmm_db.shape)

In [ ]:
gmm_db.to_csv(GMM_DB, index=False)